# CSET347 Minor Project — Topic 4
## Deep Learning for Community Detection: DMoN vs. a Heterophily-Aware Variant (DMoN-H)

**Purpose:** Reproducible starter experiment following the Topic 4 execution plan in the supplied *Minor Project Research Pipeline* (pages 29–32).

**Important:** Run every cell from top to bottom in a fresh Google Colab runtime. The notebook records the actual results it produces; do not copy or claim unrun results. This is a starter implementation, not a guarantee of publication-grade reproduction.

## 1. Research question and hypothesis

**Research question:** Does adding a node's own feature representation to a neighbourhood-based graph representation improve deep community detection, particularly on heterophilic graphs?

**Hypothesis to test:** The DMoN-H variant may be more robust than standard DMoN when connected nodes are often dissimilar. This is a hypothesis, not a result.

**Method:** Standard DMoN uses a GCN node encoder and DMoN pooling. DMoN-H concatenates the GCN representation with a separate MLP representation of each node's own features before DMoN pooling.

**Datasets:** Cora (Planetoid) and Chameleon (WikipediaNetwork). Dataset download availability and compatibility depend on the runtime.

In [ ]:
# Install dependencies in Google Colab.
# If installation finishes with a restart prompt, restart the runtime and run cells again.
%pip -q install torch-geometric scikit-learn pandas matplotlib networkx scipy

In [ ]:
import os, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score, f1_score
from scipy.optimize import linear_sum_assignment
from torch_geometric.datasets import Planetoid, WikipediaNetwork
from torch_geometric.nn import GCNConv, DMoNPooling
from torch_geometric.utils import to_dense_adj

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

## 2. Reproducibility and evaluation

Labels are used **only to evaluate** the clusters. They are not passed into the training loss, and no model selection should use label-based scores. Cluster IDs are arbitrary, so ACC uses Hungarian matching before comparison with ground-truth labels.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def cluster_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    true_ids = np.unique(y_true)
    pred_ids = np.unique(y_pred)
    mat = np.zeros((len(pred_ids), len(true_ids)), dtype=np.int64)
    pred_map = {v:i for i,v in enumerate(pred_ids)}
    true_map = {v:i for i,v in enumerate(true_ids)}
    for p,t in zip(y_pred, y_true):
        mat[pred_map[p], true_map[t]] += 1
    r,c = linear_sum_assignment(-mat)
    mapping = {pred_ids[rr]: true_ids[cc] for rr,cc in zip(r,c)}
    mapped = np.array([mapping.get(p, true_ids[0]) for p in y_pred])
    acc = float(np.mean(mapped == y_true))
    macro_f1 = f1_score(y_true, mapped, average="macro", zero_division=0)
    return acc, float(macro_f1)

def evaluate_labels(y_true, y_pred):
    acc, macro_f1 = cluster_accuracy(y_true, y_pred)
    return {
        "ACC": acc,
        "NMI": float(normalized_mutual_info_score(y_true, y_pred)),
        "ARI": float(adjusted_rand_score(y_true, y_pred)),
        "Macro_F1": macro_f1
    }

def edge_homophily(edge_index, labels, num_nodes):
    src, dst = edge_index.cpu().numpy()
    y = labels.cpu().numpy()
    mask = src != dst
    src, dst = src[mask], dst[mask]
    return float(np.mean(y[src] == y[dst])) if len(src) else float("nan")

## 3. Load datasets

The supplied guide lists Cora and Chameleon as candidate datasets. Cora is the first smoke test; Chameleon is the heterophilic benchmark. Dataset labels are kept out of model training.

In [ ]:
DATA_ROOT = "./data"
datasets = {}
load_errors = {}

try:
    datasets["Cora"] = Planetoid(DATA_ROOT, name="Cora")
    print("Loaded Cora")
except Exception as e:
    load_errors["Cora"] = repr(e)
    print("Could not load Cora:", repr(e))

try:
    datasets["Chameleon"] = WikipediaNetwork(DATA_ROOT, name="chameleon")
    print("Loaded Chameleon")
except Exception as e:
    load_errors["Chameleon"] = repr(e)
    print("Could not load Chameleon:", repr(e))

if not datasets:
    raise RuntimeError("No dataset loaded. Check internet access in Colab and rerun this cell.")

dataset_summary = []
for name, ds in datasets.items():
    d = ds[0]
    dataset_summary.append({
        "dataset": name,
        "nodes": int(d.num_nodes),
        "edges_directed": int(d.edge_index.shape[1]),
        "features": int(d.num_features),
        "classes": int(torch.unique(d.y).numel()),
        "edge_homophily": edge_homophily(d.edge_index, d.y, d.num_nodes)
    })
dataset_summary = pd.DataFrame(dataset_summary)
display(dataset_summary)

## 4. Deep clustering models

This implementation follows the guide's design: GCN neighbourhood representation; optional MLP ego representation; DMoN pooling objective. DMoNPooling's assignment matrix is used to obtain predicted cluster IDs. The dense adjacency implementation is intended for small/medium graphs that fit available memory.

In [ ]:
class DMoNModel(nn.Module):
    def __init__(self, in_dim, num_clusters, hidden=64, hetero_aware=False):
        super().__init__()
        self.hetero_aware = hetero_aware
        self.gcn = GCNConv(in_dim, hidden)
        if hetero_aware:
            self.ego = nn.Linear(in_dim, hidden)
            pool_dim = 2 * hidden
        else:
            pool_dim = hidden
        self.pool = DMoNPooling(pool_dim, num_clusters)

    def forward(self, x, edge_index, adj):
        z_neigh = F.selu(self.gcn(x, edge_index))
        if self.hetero_aware:
            z_ego = F.selu(self.ego(x))
            z = torch.cat([z_neigh, z_ego], dim=-1)
        else:
            z = z_neigh
        s, pooled_x, pooled_adj, spectral_loss, ortho_loss, cluster_loss = self.pool(
            z.unsqueeze(0), adj
        )
        loss = spectral_loss + ortho_loss + cluster_loss
        return s.squeeze(0), loss

def train_deep_model(dataset, seed=0, hetero_aware=False, epochs=150, hidden=64, lr=1e-3):
    set_seed(seed)
    data = dataset[0].to(DEVICE)
    # DMoNPooling uses dense adjacency; guard against unexpectedly large graphs.
    if data.num_nodes > 12000:
        raise ValueError(f"Dense adjacency may be too large for {data.num_nodes} nodes.")
    adj = to_dense_adj(data.edge_index, max_num_nodes=data.num_nodes).to(DEVICE)
    model = DMoNModel(data.num_features, int(torch.unique(data.y).numel()),
                      hidden=hidden, hetero_aware=hetero_aware).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    start = time.time()
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        assignments, loss = model(data.x, data.edge_index, adj)
        loss.backward()
        optimizer.step()
    elapsed = time.time() - start
    model.eval()
    with torch.no_grad():
        assignments, final_loss = model(data.x, data.edge_index, adj)
        pred = assignments.argmax(dim=-1).detach().cpu().numpy()
    metrics = evaluate_labels(data.y.detach().cpu().numpy(), pred)
    metrics.update({"loss": float(final_loss.detach().cpu()), "runtime_sec": elapsed})
    return metrics, pred

## 5. Classical baseline: NetworkX Louvain

Louvain is a classical modularity-based community detection baseline. It does not use node features. Its output is evaluated against labels using the same metrics. This gives a useful comparison, but it is not a substitute for reproducing every baseline in the final selected base paper.

In [ ]:
def run_louvain(dataset, seed=0):
    data = dataset[0]
    G = nx.Graph()
    G.add_nodes_from(range(int(data.num_nodes)))
    edges = data.edge_index.cpu().numpy().T
    G.add_edges_from((int(u), int(v)) for u,v in edges if u != v)
    start = time.time()
    try:
        communities = nx.community.louvain_communities(G, seed=seed)
    except Exception:
        communities = nx.community.greedy_modularity_communities(G)
    runtime = time.time() - start
    pred = np.full(data.num_nodes, -1, dtype=int)
    for cid, comm in enumerate(communities):
        for node in comm:
            pred[node] = cid
    metrics = evaluate_labels(data.y.cpu().numpy(), pred)
    metrics["runtime_sec"] = runtime
    return metrics, pred

## 6. Run experiments

For a fast first run, use seeds 0–2 and 100 epochs. The guide specifies five seeds (0–4), so after confirming that the notebook works, change `SEEDS` to `range(5)` and increase epochs if compute permits. Report the actual settings used.

In [ ]:
SEEDS = [0, 1, 2]  # For the full protocol use [0, 1, 2, 3, 4]
EPOCHS = 100       # Increase after the smoke test if time/GPU allows
rows = []
predictions = {}

for dataset_name, ds in datasets.items():
    print("\nDataset:", dataset_name)
    for seed in SEEDS:
        for method, kwargs in [
            ("DMoN", {"hetero_aware": False}),
            ("DMoN-H", {"hetero_aware": True})
        ]:
            try:
                metrics, pred = train_deep_model(ds, seed=seed, epochs=EPOCHS, **kwargs)
                rows.append({"dataset": dataset_name, "method": method, "seed": seed, **metrics})
                predictions[(dataset_name, method, seed)] = pred
                print(method, seed, {k: round(v, 4) for k,v in metrics.items()})
            except Exception as e:
                print("FAILED", dataset_name, method, seed, repr(e))
                rows.append({"dataset": dataset_name, "method": method, "seed": seed, "error": repr(e)})
        try:
            metrics, pred = run_louvain(ds, seed=seed)
            rows.append({"dataset": dataset_name, "method": "Louvain", "seed": seed, **metrics})
            predictions[(dataset_name, "Louvain", seed)] = pred
        except Exception as e:
            print("Louvain failed:", repr(e))

runs = pd.DataFrame(rows)
display(runs)
os.makedirs("results", exist_ok=True)
runs.to_csv("results/runs.csv", index=False)
print("Saved results/runs.csv")

## 7. Aggregate results (mean ± standard deviation)

Do not interpret missing or failed runs as zero. This summary excludes failed runs and reports the number of successful runs per method/dataset.

In [ ]:
metric_cols = ["ACC", "NMI", "ARI", "Macro_F1", "runtime_sec"]
valid = runs.copy()
for col in metric_cols:
    if col in valid:
        valid[col] = pd.to_numeric(valid[col], errors="coerce")

summary_rows = []
for (dataset_name, method), group in valid.groupby(["dataset", "method"]):
    row = {"dataset": dataset_name, "method": method, "successful_runs": int(group["NMI"].notna().sum())}
    for metric in metric_cols:
        if metric in group and group[metric].notna().any():
            row[metric + "_mean"] = group[metric].mean()
            row[metric + "_std"] = group[metric].std(ddof=1)
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
display(summary.round(4))
summary.to_csv("results/summary.csv", index=False)

## 8. Visualise the comparison

The following plot shows NMI, one of the clustering evaluation metrics listed in the guide. It uses measured runs from this notebook.

In [ ]:
plot_df = valid.dropna(subset=["NMI"])
if len(plot_df):
    ax = plot_df.groupby(["dataset", "method"])["NMI"].mean().unstack().plot(kind="bar", figsize=(9,5))
    ax.set_ylabel("Mean NMI")
    ax.set_title("Community detection comparison (actual notebook runs)")
    ax.set_ylim(0, 1)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("No successful NMI results to plot.")

## 9. Edge-removal robustness (illustrative extension)

The project guide asks for robustness under 10–50% edge removal. This cell provides a simple Louvain-only first pass to validate the protocol. A complete study should retrain DMoN and DMoN-H for every perturbation and seed, which can be expensive; do not describe this partial extension as a full deep-model robustness experiment.

In [ ]:
def perturbed_edge_index(edge_index, drop_fraction, seed):
    rng = np.random.default_rng(seed)
    edges = edge_index.cpu().numpy().T
    keep = rng.random(len(edges)) >= drop_fraction
    kept = edges[keep]
    return torch.tensor(kept.T, dtype=torch.long)

def louvain_on_edge_index(num_nodes, edge_index, labels, seed=0):
    G = nx.Graph()
    G.add_nodes_from(range(num_nodes))
    G.add_edges_from((int(u), int(v)) for u,v in edge_index.T if u != v)
    t0 = time.time()
    comms = nx.community.louvain_communities(G, seed=seed)
    pred = np.full(num_nodes, -1, dtype=int)
    for cid, comm in enumerate(comms):
        for node in comm:
            pred[node] = cid
    result = evaluate_labels(labels.cpu().numpy(), pred)
    result["runtime_sec"] = time.time() - t0
    return result

robust_rows = []
for name, ds in datasets.items():
    data = ds[0]
    for drop in [0.1, 0.2, 0.3, 0.4, 0.5]:
        for seed in SEEDS:
            eidx = perturbed_edge_index(data.edge_index, drop, seed)
            m = louvain_on_edge_index(data.num_nodes, eidx, data.y, seed)
            robust_rows.append({"dataset": name, "method": "Louvain", "edge_drop": drop, "seed": seed, **m})
robust = pd.DataFrame(robust_rows)
display(robust.groupby(["dataset", "edge_drop"])["NMI"].agg(["mean","std"]).reset_index().round(4))
robust.to_csv("results/robustness_louvain.csv", index=False)

## 10. Limitations and interpretation

- The proposed DMoN-H model is a small experimental variant of the design in the supplied guide; its performance must be established empirically.
- Dense adjacency can become a memory bottleneck. This notebook intentionally targets small/medium graphs.
- The quick-run defaults use three seeds and 100 epochs. For the full protocol, use five seeds (0–4), document the compute environment, and report any failed runs.
- The simple robustness section currently evaluates Louvain only. For full compliance with the guide, retrain both deep models under each edge perturbation.
- Label-based metrics are for evaluation only. Do not tune epochs or hyperparameters using ACC/NMI/ARI/F1.
- Cite the base paper and dataset papers after verifying the full references and protocol. The guide names DGCLUSTER (AAAI 2024) and DMoN (JMLR 2023) as candidate base papers; confirm with the instructor/team which one is finalized.

## 11. Exportable outputs

The notebook writes:
- `results/runs.csv` — one row per method/dataset/seed
- `results/summary.csv` — mean and standard deviation
- `results/robustness_louvain.csv` — preliminary Louvain edge-removal experiment

Download these from Colab's file panel after successful execution. In your report, include the exact settings used and only results present in these files.